# sphinxcontrib_jsmath

sphinxcontrib-jsmath 2.0.0 is the Sphinx extension that renders maths with the jsMath JavaScript library. It needs
`jsmath_path` (where the site serves jsMath); it does not ship jsMath itself. These cells render inline and display
maths, check the script tag it adds, and the error when `jsmath_path` is missing.

In [ ]:
# Example 1: inline maths becomes a jsMath span.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Demo'\nextensions = ['sphinxcontrib.jsmath']\njsmath_path = 'jsMath/easy/load.js'\nhtml_math_renderer = 'jsmath'\n")
(src / "index.rst").write_text("Demo\n====\n\n.. toctree::\n\n   page\n\nEuler: :math:`e^{i\\pi} + 1 = 0`.\n")
(src / "page.rst").write_text("Page\n====\n\nA page about :index:`widgets`.\n")
rc = build_main(["-q", "-W", "-b", 'html', str(src), str(src / "_out")])
out = src / "_out"
print("rc", rc, sorted(p.name for p in out.iterdir()))
assert rc == 0
html = (out / 'index.html').read_text()
assert '<span class="math notranslate nohighlight">e^{i\\pi}' in html

In [ ]:
# Example 2: the extension adds the configured jsMath loader script to the page.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Demo'\nextensions = ['sphinxcontrib.jsmath']\njsmath_path = 'jsMath/easy/load.js'\nhtml_math_renderer = 'jsmath'\n")
(src / "index.rst").write_text("Demo\n====\n\n.. toctree::\n\n   page\n\nInline :math:`x`.\n")
(src / "page.rst").write_text("Page\n====\n\nA page about :index:`widgets`.\n")
rc = build_main(["-q", "-W", "-b", 'html', str(src), str(src / "_out")])
out = src / "_out"
print("rc", rc, sorted(p.name for p in out.iterdir()))
assert rc == 0
html = (out / 'index.html').read_text()
assert 'jsMath/easy/load.js' in html

In [ ]:
# Example 3: a display equation becomes a jsMath div.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Demo'\nextensions = ['sphinxcontrib.jsmath']\njsmath_path = 'jsMath/easy/load.js'\nhtml_math_renderer = 'jsmath'\n")
(src / "index.rst").write_text("Demo\n====\n\n.. toctree::\n\n   page\n\n.. math::\n\n   \\int_0^1 x\\,dx = \\frac{1}{2}\n\n")
(src / "page.rst").write_text("Page\n====\n\nA page about :index:`widgets`.\n")
rc = build_main(["-q", "-W", "-b", 'html', str(src), str(src / "_out")])
out = src / "_out"
print("rc", rc, sorted(p.name for p in out.iterdir()))
assert rc == 0
html = (out / 'index.html').read_text()
assert '<div class="math notranslate nohighlight">' in html or '<div class="math">' in html, html[html.find('math') - 80:][:300]

In [ ]:
# Example 4: the extension's registration.
import sphinxcontrib.jsmath as jsmath

print(jsmath.__version__)
assert jsmath.__version__ == "2.0.0"
assert callable(jsmath.setup) and callable(jsmath.html_visit_math)

In [ ]:
# Example 5: without jsmath_path the build stops with the extension's own message, not a broken page.
import io, pathlib, tempfile
from contextlib import redirect_stderr
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Demo'\nextensions = ['sphinxcontrib.jsmath']\nhtml_math_renderer = 'jsmath'\n")
(src / "index.rst").write_text("Demo\n====\n\nEuler: :math:`x`.\n")
err = io.StringIO()
with redirect_stderr(err):
    rc = build_main(["-q", "-b", "html", str(src), str(src / "_out")])
print("rc", rc)
print(err.getvalue()[-400:])
assert rc != 0 and "jsmath_path" in err.getvalue()